### **Feature Extraction : -**

In [28]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# import dask.dataframe as dd

# df_t = dd.read_parquet("transactions/batch-*/*.parquet")
# df_ta= dd.read_parquet("transactions_additional/batch-*/*.parquet")

In [5]:
import duckdb

con = duckdb.connect()

result = con.execute("""
SELECT *
FROM read_parquet('transactions_additional/**/*.parquet')
LIMIT 6
""").fetchdf()

result

,transaction_id,mnemonic_code,latitude,longitude,ip_address,balance_after_transaction,part_transaction_type,atm_deposit_channel_code,transaction_sub_type
0,TXN_0000000000,END,NaN,NaN,122.175.102.100,53657.59,CI,None,normal
1,TXN_0000000001,UPD,15.786843,78.077894,103.92.190.204,53479.10,CI,None,normal
2,TXN_0000000002,CCL,NaN,NaN,None,52479.10,CI,None,normal
3,TXN_0000000003,UPC,15.781886,78.053709,122.124.163.67,59479.10,CI,None,normal
4,TXN_0000000004,FTD,NaN,NaN,103.241.164.38,64179.10,CI,None,normal
5,TXN_0000000005,END,15.831904,78.031194,157.166.43.104,154311.54,CI,None,normal


### Feature_extraction from transaction : -

In [ ]:
# con = duckdb.connect()

# df_txn = con.execute("""
# SELECT 
#     account_id,
#     COUNT(transaction_id) AS total_transactions
# FROM read_parquet('transactions/**/*.parquet')
# GROUP BY account_id
# """).fetchdf()

# df_txn.head()

,account_id,total_transactions
0,ACCT_001054,3672
1,ACCT_001059,3053
2,ACCT_001064,2058
3,ACCT_001076,3849
4,ACCT_001087,1342


In [ ]:
# con.execute("""
# COPY (
#     SELECT 
#         account_id,
#         COUNT(transaction_id) AS total_transactions
#     FROM read_parquet('transactions/**/*.parquet')
#     GROUP BY account_id
# ) TO 'account_txn.parquet' (FORMAT PARQUET)
# """)

In [13]:

result = con.execute("""
SELECT COUNT(DISTINCT mcc_code)
FROM read_parquet('transactions/**/*.parquet')
""").fetchone()

print("Unique MCC codes:", result[0])

Unique MCC codes: 248


In [15]:

result = con.execute("""
SELECT COUNT(DISTINCT counterparty_id)
FROM read_parquet('transactions/**/*.parquet')
""").fetchone()

print("Unique counterparty IDs:", result[0])

Unique counterparty IDs: 140957


In [ ]:
# import duckdb

# con = duckdb.connect()

# df_mcc = con.execute("""
# WITH mcc_counts AS (
#     SELECT 
#         account_id,
#         mcc_code,
#         COUNT(*) AS txn_count
#     FROM read_parquet('transactions/**/*.parquet')
#     GROUP BY account_id, mcc_code
# ),

# ranked AS (
#     SELECT *,
#            ROW_NUMBER() OVER (
#                PARTITION BY account_id 
#                ORDER BY txn_count DESC
#            ) AS max_rank,
           
#            ROW_NUMBER() OVER (
#                PARTITION BY account_id 
#                ORDER BY txn_count ASC
#            ) AS min_rank
#     FROM mcc_counts
# )

# SELECT 
#     account_id,
#     COUNT(DISTINCT mcc_code) AS unique_mcc_count,
#     MAX(CASE WHEN max_rank = 1 THEN mcc_code END) AS most_common_mcc,
#     MAX(CASE WHEN min_rank = 1 THEN mcc_code END) AS least_common_mcc
# FROM ranked
# GROUP BY account_id
# """).fetchdf()

# df_mcc.head()

,account_id,unique_mcc_count,most_common_mcc,least_common_mcc
0,ACCT_095495,52,9384,6211
1,ACCT_095636,53,9384,6300
2,ACCT_095855,56,9384,4816
3,ACCT_096029,47,9384,5944
4,ACCT_096167,38,9384,6011


In [18]:
df_mcc.shape[0]

159776

In [ ]:
# con.execute("""
# COPY (
#     SELECT 
#         a.account_id,
#         a.total_transactions,
#         b.unique_mcc_count,
#         b.most_common_mcc,
#         b.least_common_mcc
#     FROM read_parquet('account_txn.parquet') a
#     LEFT JOIN df_mcc b
#     USING(account_id)
# ) TO 'account_txn_features.parquet' (FORMAT PARQUET)
# """)

In [ ]:
import dask.dataframe as dd
df_txn = dd.read_parquet('account_txn_features.parquet')
df_txn.head()

,account_id,total_transactions,unique_mcc_count,most_common_mcc,least_common_mcc
0,ACCT_086880,1754,52,9384,6300
1,ACCT_087066,2839,57,9384,5541
2,ACCT_087279,523,37,9384,7399
3,ACCT_087467,1804,53,9384,5251
4,ACCT_087484,1800,55,9384,7399


In [24]:
df_txn.shape[0].compute()


159776

### 3 : -

In [31]:
import duckdb

con = duckdb.connect()

con.execute("SET threads=4")
con.execute("SET preserve_insertion_order=false")
con.execute("SET memory_limit='10GB'")

# **Columns : -**
### account_id , total_transactions ,unique_mcc_count ,most_common_mcc ,max_mcc_frequency ,mcc_max_amount ,mcc_min_amount

In [ ]:
# import duckdb

# con = duckdb.connect()

# con.execute("""
# COPY (

# WITH base AS (
#     SELECT account_id, mcc_code, amount
#     FROM read_parquet('transactions/**/*.parquet')
# ),

# -- total transactions + unique MCC
# txn_stats AS (
#     SELECT
#         account_id,
#         COUNT(*) AS total_transactions,
#         COUNT(DISTINCT mcc_code) AS unique_mcc_count,
#         arg_max(mcc_code, amount) AS mcc_max_amount,
#         arg_min(mcc_code, amount) AS mcc_min_amount
#     FROM base
#     GROUP BY account_id
# ),

# -- MCC frequency per account
# mcc_freq AS (
#     SELECT
#         account_id,
#         mcc_code,
#         COUNT(*) AS freq
#     FROM base
#     GROUP BY account_id, mcc_code
# ),

# -- most common MCC
# mcc_rank AS (
#     SELECT
#         account_id,
#         arg_max(mcc_code, freq) AS most_common_mcc,
#         MAX(freq) AS max_mcc_frequency
#     FROM mcc_freq
#     GROUP BY account_id
# )

# SELECT
#     t.account_id,
#     t.total_transactions,
#     t.unique_mcc_count,
#     r.most_common_mcc,
#     r.max_mcc_frequency,
#     t.mcc_max_amount,
#     t.mcc_min_amount
# FROM txn_stats t
# LEFT JOIN mcc_rank r
# USING(account_id)

# ) TO 'account_features.parquet'
# (FORMAT PARQUET)
# """)

In [10]:
import dask.dataframe as dd
df_txn = dd.read_parquet('account_features.parquet')
df_txn.head()

,account_id,total_transactions,unique_mcc_count,most_common_mcc,max_mcc_frequency,mcc_max_amount,mcc_min_amount
0,ACCT_161342,6114,58,9384,1262,2017,5651
1,ACCT_126197,946,52,9384,186,5905,9384
2,ACCT_154220,89,19,9384,20,2921,6051
3,ACCT_016873,102,18,5682,21,2017,5651
4,ACCT_170087,2046,52,9384,396,6051,5682


## **3 Channel : -**
### account_id, unique_channel_count, most_common_channel, max_channel_frequency, channel_max_amount, channel_min_amount

In [ ]:
# import duckdb

# con = duckdb.connect()

# con.execute("""
# COPY (

# WITH base AS (
#     SELECT account_id, channel, amount
#     FROM read_parquet('transactions/**/*.parquet')
# ),

# txn_stats AS (
#     SELECT
#         account_id,
#         COUNT(DISTINCT channel) AS unique_channel_count,
#         arg_max(channel, amount) AS channel_max_amount,
#         arg_min(channel, amount) AS channel_min_amount
#     FROM base
#     GROUP BY account_id
# ),

# channel_freq AS (
#     SELECT
#         account_id,
#         channel,
#         COUNT(*) AS freq
#     FROM base
#     GROUP BY account_id, channel
# ),

# channel_rank AS (
#     SELECT
#         account_id,
#         arg_max(channel, freq) AS most_common_channel,
#         MAX(freq) AS max_channel_frequency
#     FROM channel_freq
#     GROUP BY account_id
# )

# SELECT
#     t.account_id,
#     t.unique_channel_count,
#     r.most_common_channel,
#     r.max_channel_frequency,
#     t.channel_max_amount,
#     t.channel_min_amount
# FROM txn_stats t
# LEFT JOIN channel_rank r
# USING(account_id)

# ) TO 'account_channel_features.parquet'
# (FORMAT PARQUET)
# """)

In [35]:
df_txn = dd.read_parquet('account_channel_features.parquet')
df_txn.head()

,account_id,unique_channel_count,most_common_channel,max_channel_frequency,channel_max_amount,channel_min_amount
0,ACCT_013963,24,UPC,161,UPC,OCD
1,ACCT_023593,34,UPC,371,UPD,NTD
2,ACCT_023909,24,UPD,118,UPC,UPD
3,ACCT_038801,35,UPC,823,UPD,UPC
4,ACCT_047656,33,UPC,757,TPD,UPC


In [ ]:
# import duckdb

# con = duckdb.connect()

# # tune memory/threads for your 16GB laptop (adjust if needed)
# con.execute("SET threads=4;")
# con.execute("SET memory_limit='10GB';")

# con.execute("""
# COPY (
# WITH base AS (
#   SELECT
#     account_id,
#     mcc_code,
#     CAST(transaction_timestamp AS TIMESTAMP) AS ts
#   FROM read_parquet('transactions/**/*.parquet')
# ),

# -- per-account per-hour distinct MCC count
# per_hour AS (
#   SELECT
#     account_id,
#     date_trunc('hour', ts) AS hour,
#     COUNT(DISTINCT mcc_code) AS unique_mcc_hour
#   FROM base
#   GROUP BY account_id, hour
# ),

# -- per-account per-day distinct MCC count
# per_day AS (
#   SELECT
#     account_id,
#     date_trunc('day', ts) AS day,
#     COUNT(DISTINCT mcc_code) AS unique_mcc_day
#   FROM base
#   GROUP BY account_id, day
# ),

# -- average of the hourly distinct-MCC counts per account
# hour_avg AS (
#   SELECT
#     account_id,
#     AVG(unique_mcc_hour)::DOUBLE AS avg_unique_mcc_per_hour,
#     COUNT(*) AS hours_observed
#   FROM per_hour
#   GROUP BY account_id
# ),

# -- average of the daily distinct-MCC counts per account
# day_avg AS (
#   SELECT
#     account_id,
#     AVG(unique_mcc_day)::DOUBLE AS avg_unique_mcc_per_day,
#     COUNT(*) AS days_observed
#   FROM per_day
#   GROUP BY account_id
# )

# SELECT
#   h.account_id,
#   h.avg_unique_mcc_per_hour,
#   h.hours_observed,
#   d.avg_unique_mcc_per_day,
#   d.days_observed
# FROM hour_avg h
# LEFT JOIN day_avg d USING(account_id)

# ) TO 'account_mcc_hour_day_avgs.parquet' (FORMAT PARQUET)
# """)

In [37]:
df_txn = dd.read_parquet('account_mcc_hour_day_avgs.parquet')
df_txn.head()

,account_id,avg_unique_mcc_per_hour,hours_observed,avg_unique_mcc_per_day,days_observed
0,ACCT_018391,1.178739,2126,4.424691,405
1,ACCT_089913,1.031824,1634,1.507150,1049
2,ACCT_143579,1.104428,2710,3.096569,787
3,ACCT_012905,1.203598,3669,4.824561,627
4,ACCT_088243,1.119371,2798,3.342466,730


In [3]:
import pandas as pd

df = pd.read_parquet("account_mcc_hour_day_avgs.parquet")

df["avg_unique_mcc_per_day"] = df["avg_unique_mcc_per_day"].astype("int64")
df["avg_unique_mcc_per_hour"] = df["avg_unique_mcc_per_hour"].astype("int64")

df = df.drop(columns=["hours_observed", "days_observed"])

df.head()


,account_id,avg_unique_mcc_per_hour,avg_unique_mcc_per_day
0,ACCT_018391,1,4
1,ACCT_089913,1,1
2,ACCT_143579,1,3
3,ACCT_012905,1,4
4,ACCT_088243,1,3


In [4]:
df.to_parquet(
    "account_mcc_hour_day_avgs.parquet",
    index=False
)

In [5]:
import pandas as pd

df = pd.read_parquet("account_mcc_hour_day_avgs.parquet")
df.head()

,account_id,avg_unique_mcc_per_hour,avg_unique_mcc_per_day
0,ACCT_018391,1,4
1,ACCT_089913,1,1
2,ACCT_143579,1,3
3,ACCT_012905,1,4
4,ACCT_088243,1,3


In [38]:
import duckdb

con = duckdb.connect()

con.execute("SET threads=4;")
con.execute("SET memory_limit='10GB';")

con.execute("""
COPY (
WITH base AS (
  SELECT
    account_id,
    channel,
    CAST(transaction_timestamp AS TIMESTAMP) AS ts
  FROM read_parquet('transactions/**/*.parquet')
),

per_hour AS (
  SELECT
    account_id,
    date_trunc('hour', ts) AS hour,
    COUNT(DISTINCT channel) AS unique_channel_hour
  FROM base
  GROUP BY account_id, hour
),

per_day AS (
  SELECT
    account_id,
    date_trunc('day', ts) AS day,
    COUNT(DISTINCT channel) AS unique_channel_day
  FROM base
  GROUP BY account_id, day
),

hour_avg AS (
  SELECT
    account_id,
    CAST(ROUND(AVG(unique_channel_hour)) AS INTEGER) AS avg_unique_channel_per_hour,
    COUNT(*) AS hours_observed
  FROM per_hour
  GROUP BY account_id
),

day_avg AS (
  SELECT
    account_id,
    CAST(ROUND(AVG(unique_channel_day)) AS INTEGER) AS avg_unique_channel_per_day,
    COUNT(*) AS days_observed
  FROM per_day
  GROUP BY account_id
)

SELECT
  h.account_id,
  h.avg_unique_channel_per_hour,
  h.hours_observed,
  d.avg_unique_channel_per_day,
  d.days_observed
FROM hour_avg h
LEFT JOIN day_avg d USING(account_id)

) TO 'account_channel_hour_day_avgs.parquet' (FORMAT PARQUET)
""")

In [6]:
import pandas as pd
df = pd.read_parquet("account_channel_hour_day_avgs.parquet")
df.head()

,account_id,avg_unique_channel_per_hour,hours_observed,avg_unique_channel_per_day,days_observed
0,ACCT_060997,1,1780,1,1066
1,ACCT_126480,1,8980,6,904
2,ACCT_178495,1,2457,2,744
3,ACCT_008371,1,2226,2,674
4,ACCT_057936,1,2622,4,460


In [7]:
df = df.drop(columns=["hours_observed", "days_observed"])
df.head()

,account_id,avg_unique_channel_per_hour,avg_unique_channel_per_day
0,ACCT_060997,1,1
1,ACCT_126480,1,6
2,ACCT_178495,1,2
3,ACCT_008371,1,2
4,ACCT_057936,1,4


In [8]:
df.to_parquet(
    "account_channel_hour_day_avgs.parquet",
    index=False
)

In [ ]:
df = pd.read_parquet("account_channel_hour_day_avgs.parquet")
df.head()

,account_id,avg_unique_channel_per_hour,avg_unique_channel_per_day
0,ACCT_060997,1,1
1,ACCT_126480,1,6
2,ACCT_178495,1,2
3,ACCT_008371,1,2
4,ACCT_057936,1,4


In [12]:
import duckdb

con = duckdb.connect()

con.execute("""
COPY (
SELECT
    base.*,
    ch.* EXCLUDE(account_id),
    mcc.* EXCLUDE(account_id),
    chd.* EXCLUDE(account_id)

FROM 'account_features.parquet' base

LEFT JOIN 'account_channel_features.parquet' ch
ON base.account_id = ch.account_id

LEFT JOIN 'account_mcc_hour_day_avgs.parquet' mcc
ON base.account_id = mcc.account_id

LEFT JOIN 'account_channel_hour_day_avgs.parquet' chd
ON base.account_id = chd.account_id
)
TO 'transaction_features.parquet'
(FORMAT PARQUET)
""")

In [17]:
import pandas as pd
df = pd.read_parquet('transaction_features.parquet')
df.head()

,account_id,total_transactions,unique_mcc_count,most_common_mcc,max_mcc_frequency,mcc_max_amount,mcc_min_amount,unique_channel_count,most_common_channel,max_channel_frequency,channel_max_amount,channel_min_amount,avg_unique_mcc_per_hour,avg_unique_mcc_per_day,avg_unique_channel_per_hour,avg_unique_channel_per_day
0,ACCT_013963,442,39,9384,83,5651,9384,24,UPC,161,UPC,OCD,1,1,1,1
1,ACCT_023593,894,48,9384,189,2017,9384,34,UPC,371,UPD,NTD,1,1,1,2
2,ACCT_023909,295,33,9384,54,1139,2557,24,UPD,118,UPC,UPD,1,1,1,1
3,ACCT_038801,2113,55,9384,447,2557,2017,35,UPC,823,UPD,UPC,1,2,1,2
4,ACCT_047656,1988,55,9384,394,2557,5651,33,UPC,757,TPD,UPC,1,1,1,2


In [18]:
df.shape

(159776, 16)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159776 entries, 0 to 159775
Data columns (total 16 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   account_id                   159776 non-null  object
 1   total_transactions           159776 non-null  int64 
 2   unique_mcc_count             159776 non-null  int64 
 3   most_common_mcc              159776 non-null  int64 
 4   max_mcc_frequency            159776 non-null  int64 
 5   mcc_max_amount               159776 non-null  int64 
 6   mcc_min_amount               159776 non-null  int64 
 7   unique_channel_count         159776 non-null  int64 
 8   most_common_channel          159776 non-null  object
 9   max_channel_frequency        159776 non-null  int64 
 10  channel_max_amount           159776 non-null  object
 11  channel_min_amount           159776 non-null  object
 12  avg_unique_mcc_per_hour      159776 non-null  int64 
 13  avg_unique_mcc

In [20]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE TABLE account_least_features AS

WITH base AS (
    SELECT
        account_id,
        channel,
        mcc_code
    FROM read_parquet('transactions/**/*.parquet')
),

channel_counts AS (
    SELECT
        account_id,
        channel,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY COUNT(*) ASC
        ) AS rn
    FROM base
    GROUP BY account_id, channel
),

mcc_counts AS (
    SELECT
        account_id,
        mcc_code,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY COUNT(*) ASC
        ) AS rn
    FROM base
    GROUP BY account_id, mcc_code
)

SELECT
    c.account_id,
    c.channel AS least_common_channel,
    m.mcc_code AS least_common_mcc_code

FROM channel_counts c
LEFT JOIN mcc_counts m
ON c.account_id = m.account_id

WHERE c.rn = 1
AND m.rn = 1
""")

In [21]:
con.execute("""
COPY (
SELECT
    tf.*,
    lf.least_common_channel,
    lf.least_common_mcc_code

FROM 'transaction_features.parquet' tf

LEFT JOIN account_least_features lf
ON tf.account_id = lf.account_id
)
TO 'transaction_features.parquet'
(FORMAT PARQUET)
""")

In [22]:
import pandas as pd
df = pd.read_parquet('transaction_features.parquet')
df.head()

,account_id,total_transactions,unique_mcc_count,most_common_mcc,max_mcc_frequency,mcc_max_amount,mcc_min_amount,unique_channel_count,most_common_channel,max_channel_frequency,channel_max_amount,channel_min_amount,avg_unique_mcc_per_hour,avg_unique_mcc_per_day,avg_unique_channel_per_hour,avg_unique_channel_per_day,least_common_channel,least_common_mcc_code
0,ACCT_013963,442,39,9384,83,5651,9384,24,UPC,161,UPC,OCD,1,1,1,1,ATW,5933
1,ACCT_023593,894,48,9384,189,2017,9384,34,UPC,371,UPD,NTD,1,1,1,2,RTD,4900
2,ACCT_023909,295,33,9384,54,1139,2557,24,UPD,118,UPC,UPD,1,1,1,1,OCD,4814
3,ACCT_038801,2113,55,9384,447,2557,2017,35,UPC,823,UPD,UPC,1,2,1,2,CCL,4816
4,ACCT_047656,1988,55,9384,394,2557,5651,33,UPC,757,TPD,UPC,1,1,1,2,IAD,5541


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159776 entries, 0 to 159775
Data columns (total 18 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   account_id                   159776 non-null  object
 1   total_transactions           159776 non-null  int64 
 2   unique_mcc_count             159776 non-null  int64 
 3   most_common_mcc              159776 non-null  int64 
 4   max_mcc_frequency            159776 non-null  int64 
 5   mcc_max_amount               159776 non-null  int64 
 6   mcc_min_amount               159776 non-null  int64 
 7   unique_channel_count         159776 non-null  int64 
 8   most_common_channel          159776 non-null  object
 9   max_channel_frequency        159776 non-null  int64 
 10  channel_max_amount           159776 non-null  object
 11  channel_min_amount           159776 non-null  object
 12  avg_unique_mcc_per_hour      159776 non-null  int64 
 13  avg_unique_mcc

In [26]:
df.shape

(159776, 18)

In [1]:
import duckdb

con = duckdb.connect()

df_head = con.execute("""
SELECT *
FROM read_parquet('transactions/**/*.parquet')
LIMIT 5
""").df()

print("Head of dataset:")
print(df_head)

Head of dataset:
   transaction_id   account_id transaction_timestamp  mcc_code channel  \
0  TXN_0001000000  ACCT_000494   2023-05-22T22:47:35      9384     UPD   
1  TXN_0001000001  ACCT_000494   2023-05-22T22:53:31      2943     MAC   
2  TXN_0001000002  ACCT_000494   2023-05-23T01:40:04      5651     UPC   
3  TXN_0001000003  ACCT_000494   2023-05-23T05:48:59      6101     UPD   
4  TXN_0001000004  ACCT_000494   2023-05-23T10:33:24      5651     UPD   

     amount txn_type counterparty_id  
0    299.55        C       CP_090625  
1  11191.27        D       CP_070045  
2    159.42        D       CP_029345  
3  23700.00        D       CP_038556  
4   2000.00        D       CP_000291  


In [2]:
rows = con.execute("""
SELECT COUNT(*)
FROM read_parquet('transactions/**/*.parquet')
""").fetchone()[0]

print("Rows:", rows)

Rows: 396875323
